# Часть E. Наш репозиторий и форк OpenUnlearning

Проверяем в Colab то, что сделано в части E: репозиторий диплома клонируется вместе с форком OpenUnlearning, ссылки на Drive не попадают в git, окружение собирается из lock-файлов репозитория, а код форка оценивает модель M так же, как песочница части D. Нужна GPU: L4 или A100 (для оценки в шаге 8). Перед этой частью должны быть выполнены части A–D.
Подробный разбор — в файле [docs/E_explained.md](https://github.com/IvanovskyDev/Machine-Unlearning-in-LLM/blob/main/docs/E_explained.md).

**1. Старт сессии** — как шаг 1 части D, плюс путь к репозиторию `REPO`.

In [ ]:
import os                                   # папки и переменные окружения

from google.colab import drive, userdata    # Google Drive и секреты Colab

drive.mount("/content/drive")               # подключить Google Drive

DRIVE = "/content/drive/MyDrive/unlearning_data"   # папка проекта на Drive (постоянная)
FAST = "/content/fast"                             # папка на диске машины (очищается после сессии)
REPO = "/content/repo"                             # сюда шаг 3 клонирует репозиторий диплома

os.makedirs(DRIVE + "/saves", exist_ok=True)       # результаты OpenUnlearning
os.makedirs(DRIVE + "/results_raw", exist_ok=True) # сырые результаты атак
os.makedirs(DRIVE + "/logs", exist_ok=True)        # логи запусков и замеры времени и памяти
os.makedirs(FAST + "/hf_home", exist_ok=True)      # кэш Hugging Face
os.makedirs(FAST + "/models", exist_ok=True)       # скачанные модели

os.environ["BIG"] = DRIVE                          # «большой диск» из плана
os.environ["REPO"] = REPO                          # путь к репозиторию для ячеек %%bash
os.environ["HF_HOME"] = FAST + "/hf_home"          # кэш Hugging Face
os.environ["MODELS"] = FAST + "/models"            # папка моделей
os.environ["TOKENIZERS_PARALLELISM"] = "false"     # меньше лишних предупреждений
os.environ["PYTHONUNBUFFERED"] = "1"               # вывод программ сразу попадает в лог
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")  # токен Hugging Face из секрета HF_TOKEN

print("Старт сессии выполнен")

**2. Ставим uv и убираем настройки Colab, которые мешают окружениям** — как шаг 2 части D.

In [ ]:
# Colab задаёт свои переменные окружения, которые мешают нашим окружениям:
#   UV_...      — велят uv ставить пакеты в системный Python 3.13 Colab с его ограничениями версий;
#   PYTHONPATH  — подмешивает модули Colab в любой запущенный Python;
#   MPLBACKEND  — настройка графиков блокнота; в наших окружениях из-за неё падают vLLM и BERTScore.
for name in list(os.environ):
    if name.startswith("UV_") or name in ["PYTHONPATH", "MPLBACKEND"]:
        print("Убираю", name, "=", os.environ.pop(name))

# поставить uv (-q — без подробного вывода) и проверить, что он работает
!pip install -q uv
!uv --version

**3. Клонируем репозиторий диплома вместе с форком** (блок 19). Должны напечататься коммит репозитория, строка про `external/open-unlearning`, три коммита форка (два наших и `4ad738a`) и три строки с `output.logits.float()` — правка bf16 теперь часть кода форка.

In [ ]:
%%bash
set -e                                     # остановиться на первой ошибке
rm -rf $REPO                               # удалить старую копию, если есть (данные на Drive не трогаются)
# --recurse-submodules — вместе с репозиторием скачать и форк OpenUnlearning (он подключён как submodule)
git clone -q --recurse-submodules https://github.com/IvanovskyDev/Machine-Unlearning-in-LLM.git $REPO
cd $REPO
git log -1 --format='репозиторий: %h %s'  # последний коммит репозитория диплома
git submodule status                      # какой коммит форка закреплён в репозитории
cd external/open-unlearning
git log -3 --format='форк: %h %s'         # три последних коммита форка
grep -n "output.logits" src/evals/metrics/utils.py   # правка bf16 уже в коде форка

**4. Связываем папки с Drive** (блок 19): `saves` форка и `results/raw` ведут на Drive. В конце должно напечататься, что `git status` пуст: ссылки в git не попадают.

In [ ]:
%%bash
cd $REPO
ln -s $BIG/saves external/open-unlearning/saves   # результаты OpenUnlearning — на Drive, как в части D
ln -s $BIG/results_raw results/raw                # сырые результаты атак — на Drive
ls -ld external/open-unlearning/saves results/raw # показать, куда ведут ссылки
# git status --porcelain печатает изменённые и новые файлы; пустой вывод — всё чисто
if [ -z "$(git status --porcelain)" ]; then
  echo "git status пуст: ссылки на Drive в git не попадают"
else
  git status --short                              # что-то лишнее — разобраться, прежде чем идти дальше
fi

**5. Смотрим устройство репозитория** (блок 20) и сверяем lock-файлы репозитория с копией на Drive: они должны совпадать.

In [ ]:
%%bash
cd $REPO
cat .gitmodules                   # где живёт форк и за какой его веткой следит репозиторий
ls                                # папки верхнего уровня
ls envs scripts                   # lock-файлы окружений и скрипты, перенесённые из блокнотов
# cmp сравнивает два файла побайтно и молчит, если они одинаковы
for f in requirements-unl.lock requirements-atk.lock models.lock.json; do
  cmp envs/$f $BIG/envs/$f && echo "$f: совпадает с Drive"
done

**6. Собираем окружение `unl` из lock-файла репозитория** — как в шаге 4 части D, но файл берётся из `envs/` репозитория. Несколько минут; в конце должно быть `2.4.1+cu121 12.1 True <ваша GPU>` и `4.51.3 2.6.3`.

In [ ]:
%%bash
set -e
# unl: новое окружение, пакеты ровно по lock-файлу (-r — список из файла), затем FlashAttention (его в lock-файле нет)
uv venv /content/envs/unl --python 3.11 --seed --clear
source /content/envs/unl/bin/activate
uv pip install -r $REPO/envs/requirements-unl.lock
uv pip install "https://github.com/Dao-AILab/flash-attention/releases/download/v2.6.3/flash_attn-2.6.3+cu123torch2.4cxx11abiFALSE-cp311-cp311-linux_x86_64.whl"
# проверка, как в шаге 7 части B: версии torch и CUDA, видна ли GPU; версии transformers и FlashAttention
python -c "
import torch, transformers, flash_attn
print(torch.__version__, torch.version.cuda, torch.cuda.is_available(), torch.cuda.get_device_name(0))
print(transformers.__version__, flash_attn.__version__)
"

**7. Скачиваем модель M по ревизии из репозитория** — как шаг 5 части D, но ревизия берётся из `envs/models.lock.json` репозитория.

In [ ]:
import json

from huggingface_hub import snapshot_download   # скачать все файлы модели

with open(REPO + "/envs/models.lock.json") as f:
    lock = json.load(f)                          # закреплённые ревизии моделей — теперь из репозитория

repo_id = "open-unlearning/tofu_Llama-3.2-1B-Instruct_full"   # M — знает всех авторов TOFU
folder = FAST + "/models/tofu_Llama-3.2-1B-Instruct_full"
snapshot_download(repo_id=repo_id, revision=lock[repo_id], local_dir=folder)
print("Скачана:", folder, "| ревизия", lock[repo_id][:7])   # [:7] — первые 7 знаков хэша

**8. Оцениваем M кодом форка** — та же команда, что в шаге 9 части D, но из папки форка и со скриптом замера из `scripts/` репозитория. Около двух минут; итоговые числа — в `saves/eval/sandbox_fork_eval_1B_full_f01/TOFU_SUMMARY.json` на Drive.

In [ ]:
%%bash
source /content/envs/unl/bin/activate
cd $REPO/external/open-unlearning               # код форка вместо песочницы части D
M=$MODELS/tofu_Llama-3.2-1B-Instruct_full       # папка модели M
bash $REPO/scripts/measure.sh sandbox_fork_eval_1B_full_f01 \
  python src/eval.py --config-name=eval.yaml \
  experiment=eval/tofu/default \
  model=Llama-3.2-1B-Instruct \
  model.model_args.pretrained_model_name_or_path=$M \
  model.tokenizer_args.pretrained_model_name_or_path=$M \
  forget_split=forget01 holdout_split=holdout01 \
  retain_logs_path=saves/eval/tofu_Llama-3.2-1B-Instruct_retain99/TOFU_EVAL.json \
  eval.tofu.overwrite=true \
  task_name=sandbox_fork_eval_1B_full_f01

**9. Сверяем числа форка с авторами.** На A100 все метрики должны совпасть, как в части D; на L4 бывают небольшие расхождения у ROUGE, extraction_strength и FQ (разбор D, раздел 5).

In [ ]:
import json

# оценка M кодом форка и оценка той же модели авторами фреймворка (их файл скачал шаг 6 части B)
with open(DRIVE + "/saves/eval/sandbox_fork_eval_1B_full_f01/TOFU_SUMMARY.json") as f:
    ours = json.load(f)
with open(DRIVE + "/saves/eval/tofu_Llama-3.2-1B-Instruct_full/evals_forget01/TOFU_SUMMARY.json") as f:
    authors = json.load(f)

print(f"{'Метрика':22} {'M, форк':>12} {'M, авторы':>12}  Сверка")
for metric in ours:
    if metric in authors:
        difference = abs(ours[metric] - authors[metric])
        allowed = 0.001 * max(1, abs(authors[metric]))   # третий знак; для больших чисел (privleak) — от их размера
        if difference <= allowed:
            verdict = "совпадает"
        else:
            verdict = "отличается на " + str(round(difference, 4))
        print(f"{metric:22} {ours[metric]:12.4g} {authors[metric]:12.4g}  {verdict}")

**10. Записываем часть E в журнал** (`journal.md` на Drive). Строку «Наблюдения» допишите сами.

In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo

# коммиты репозитория диплома и форка; -C папка — выполнить git в этой папке
repo_commit = !git -C $REPO log -1 --format=%h
fork_commit = !git -C $REPO/external/open-unlearning log -1 --format=%h
gpu = !nvidia-smi --query-gpu=name,driver_version --format=csv,noheader
today = datetime.now(ZoneInfo("Europe/Moscow")).date()

fq = ours["forget_quality"]            # числа оценки из шага 9
mu = ours["model_utility"]
rouge = ours["forget_Q_A_ROUGE"]

text = f"""
## {today} — Часть E: репозиторий и форк OpenUnlearning (Colab)
- Где: Google Colab, {gpu[0]} (имя GPU, драйвер)
- Репозиторий: Machine-Unlearning-in-LLM {repo_commit[0]}; форк open-unlearning, ветка tau, {fork_commit[0]} (4ad738a + ignore saves + правка bf16)
- Окружение unl и ревизия модели M — из envs/ репозитория; скрипт замера — из scripts/
- Оценка M кодом форка (sandbox_fork_eval_1B_full_f01): FQ = {fq:.3g}, MU = {mu:.3f}, forget ROUGE = {rouge:.3f}
- Наблюдения: …
"""

with open(DRIVE + "/journal.md", "a", encoding="utf-8") as f:   # дописать запись в конец журнала
    f.write(text)

print(text)

**11. Завершаем сессию**: дожидаемся, пока все файлы запишутся на Drive, и отключаем его. Чтобы продолжить работу после этого шага, начните снова с шага 1.

In [ ]:
from google.colab import drive

drive.flush_and_unmount()          # записать на Drive всё, что ещё не записано, и отключить его
print("Все файлы записаны на Drive. Машину можно отключить: Runtime → Disconnect and delete runtime")